In [1]:
!wget https://huggingface.co/datasets/SprintML/tml26_task3/resolve/main/train.npz
!pip install torchattacks

--2026-06-10 23:57:58--  https://huggingface.co/datasets/SprintML/tml26_task3/resolve/main/train.npz
Resolving huggingface.co (huggingface.co)... 3.165.160.11, 3.165.160.61, 3.165.160.12, ...
Connecting to huggingface.co (huggingface.co)|3.165.160.11|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/6a1d9dd41061848fe7561d3a/a9302d6df1ee4ac026dfaa26ff81abd26a4635b6c9841549a08c70f743f5302d?Expires=1781139478&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzZhMWQ5ZGQ0MTA2MTg0OGZlNzU2MWQzYS9hOTMwMmQ2ZGYxZWU0YWMwMjZkZmFhMjZmZjgxYWJkMjZhNDYzNWI2Yzk4NDE1NDlhMDhjNzBmNzQzZjUzMDJkKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4MTEzOTQ3OH19fV19&Signature=MEUCIQDi20ge3r651qDycBlpzGoBbb%7ELpTauZviXMdnSUWqwVgIgLbQfKB1vs9dh2RTlGAQXJ8D%7E6gV6m7TRPxAqd9SbLjo_&Key-Pair-Id=K1LYXO563TGWFU&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*

In [ ]:
import argparse
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import torchvision.transforms as T
from torchvision.models import resnet18
import torchattacks
from tqdm import tqdm
from datetime import datetime
 
# Hyperparameters
EPOCHS       = 100
BATCH_SIZE   = 256
LR           = 0.1
MOMENTUM     = 0.9
WEIGHT_DECAY = 5e-4
NUM_CLASSES  = 9
 
EPS       = 8 / 255.0    # max perturbation
ALPHA     = 2 / 255.0    # PGD step size
PGD_STEPS = 7            # PGD steps during training
VAL_SPLIT = 0.05         # 5% held out for local eval
 
OUTPUT_DIR = "/kaggle/working"
MODEL_PATH = f"{OUTPUT_DIR}/model.pt"
REPORT_PATH = f"{OUTPUT_DIR}/report.txt"
API_KEY = ""
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
 
 
# Dataset
def load_data(path):
    data   = np.load(path)
    images = torch.from_numpy(data["images"]).float() / 255.0
    labels = torch.from_numpy(data["labels"]).long()
    return images, labels
 
 
class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, augment=True):
        self.images  = images
        self.labels  = labels
        self.transform = T.Compose([
            T.RandomCrop(32, padding=4),
            T.RandomHorizontalFlip(),
        ]) if augment else None
 
    def __len__(self):
        return len(self.labels)
 
    def __getitem__(self, idx):
        x = self.images[idx]
        y = self.labels[idx]
        if self.transform:
            x = self.transform(x)
        return x, y
 
 
# Model Architecture
def build_model():
    model = resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model.to(DEVICE)
 
 
# Evaluation
@torch.no_grad()
def evaluate_clean(model, loader):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (model(x).argmax(1) == y).sum().item()
        total   += y.size(0)
    model.train()
    return correct / total
 
 
def evaluate_robust(model, loader, eps, alpha, steps):
    atk = torchattacks.PGD(model, eps=eps, alpha=alpha, steps=steps)
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x_adv = atk(x, y)
        with torch.no_grad():
            correct += (model(x_adv).argmax(1) == y).sum().item()
        total += y.size(0)
    model.train()
    return correct / total
 
 
# Training
def train(data_path):
    images, labels = load_data(data_path)
    full_dataset   = AugmentedDataset(images, labels, augment=True)
 
    n_val   = int(len(full_dataset) * VAL_SPLIT)
    n_train = len(full_dataset) - n_val
    train_ds, val_ds = random_split(full_dataset, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(42))
 
    val_images = images[val_ds.indices]
    val_labels = labels[val_ds.indices]
    val_loader = DataLoader(TensorDataset(val_images, val_labels),
                            batch_size=256, shuffle=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=2, pin_memory=True)
 
    model     = build_model()
    optimizer = torch.optim.SGD(model.parameters(), lr=LR,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()
 
    best_score = 0.0
    epoch_log  = []
 
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
 
        atk  = torchattacks.PGD(model, eps=EPS, alpha=ALPHA, steps=PGD_STEPS)
        pbar = tqdm(train_loader, desc=f"Epoch {epoch:3d}/{EPOCHS}", leave=False)
 
        for x, y in pbar:
            x, y  = x.to(DEVICE), y.to(DEVICE)
            x_adv = atk(x, y)
 
            model.train()
            logits = model(x_adv)
            loss   = criterion(logits, y)
 
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
 
            total_loss += loss.item() * y.size(0)
            correct    += (logits.argmax(1) == y).sum().item()
            total      += y.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct/total:.3f}")
 
        scheduler.step()
 
        train_loss = total_loss / total
        train_acc  = correct / total
 
        if epoch % 5 == 0 or epoch == 1:
            clean_acc  = evaluate_clean(model, val_loader)
            robust_acc = evaluate_robust(model, val_loader, EPS, ALPHA, steps=20)
            score      = 0.5 * clean_acc + 0.5 * robust_acc
 
            epoch_log.append({
                "epoch": epoch, "loss": train_loss,
                "train_acc": train_acc, "clean": clean_acc,
                "robust": robust_acc, "score": score
            })
 
            print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {train_loss:.4f} | "
                  f"Train Adv Acc: {train_acc:.3f} | Val Clean: {clean_acc:.3f} | "
                  f"Val Robust: {robust_acc:.3f} | Score: {score:.3f}")
 
            if score > best_score:
                best_score = score
                torch.save(model.state_dict(), MODEL_PATH)
                print(f"  ✓ Saved best model (score={score:.4f}) → {MODEL_PATH}")
        else:
            print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {train_loss:.4f} | "
                  f"Train Adv Acc: {train_acc:.3f}")
 
    print(f"\nDone. Best score: {best_score:.4f}")
    return best_score, epoch_log
 
 
# Report
def save_report(best_score, epoch_log):
    with open(REPORT_PATH, "w") as f:
        f.write("TML 2026 Task 3 - Adversarial Robustness Training Report\n")
        f.write(f"{'='*55}\n")
        f.write(f"Date:          {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Device:        {DEVICE}\n\n")
        f.write("Hyperparameters:\n")
        f.write(f"  Architecture: ResNet18\n")
        f.write(f"  EPOCHS:       {EPOCHS}\n")
        f.write(f"  BATCH_SIZE:   {BATCH_SIZE}\n")
        f.write(f"  LR:           {LR}\n")
        f.write(f"  LR Schedule:  CosineAnnealingLR\n")
        f.write(f"  WEIGHT_DECAY: {WEIGHT_DECAY}\n")
        f.write(f"  EPS:          {EPS:.5f} ({round(EPS*255)}/255)\n")
        f.write(f"  ALPHA:        {ALPHA:.5f} ({round(ALPHA*255)}/255)\n")
        f.write(f"  PGD_STEPS:    {PGD_STEPS} (train) / 20 (eval)\n")
        f.write(f"  VAL_SPLIT:    {VAL_SPLIT}\n\n")
        f.write("Epoch Log (every 5 epochs):\n")
        f.write(f"  {'Epoch':>6} | {'Loss':>7} | {'TrainAdv':>9} | {'ValClean':>9} | {'ValRobust':>10} | {'Score':>7}\n")
        f.write(f"  {'-'*65}\n")
        for e in epoch_log:
            f.write(f"  {e['epoch']:>6} | {e['loss']:>7.4f} | {e['train_acc']:>9.4f} | "
                    f"{e['clean']:>9.4f} | {e['robust']:>10.4f} | {e['score']:>7.4f}\n")
        f.write(f"\nBest Score: {best_score:.4f}\n")
        f.write(f"Model saved to: {MODEL_PATH}\n")
    print(f"Report saved → {REPORT_PATH}")

def submit(api_key, model_path, model_name="resnet18"):
    import os, sys, requests

    BASE_URL = "http://34.63.153.158"
    TASK_ID  = "03-robustness"

    if not os.path.isfile(model_path):
        print(f"File not found: {model_path}", file=sys.stderr)
        return

    try:
        with open(model_path, "rb") as f:
            files = {"file": (os.path.basename(model_path), f, "application/x-pytorch")}
            resp  = requests.post(
                f"{BASE_URL}/submit/{TASK_ID}",
                headers={"X-API-Key": api_key},
                files=files,
                data={"model_name": model_name},
            )

        try:
            body = resp.json()
        except Exception:
            body = {"raw_text": resp.text}

        if resp.status_code == 413:
            print("Upload rejected: file too large (HTTP 413).", file=sys.stderr)
            return

        resp.raise_for_status()
        print("Successfully submitted.")
        print("Server response:", body)

    except requests.exceptions.RequestException as e:
        detail = getattr(e, "response", None)
        print(f"Submission error: {e}")
        if detail is not None:
            try:    print("Server response:", detail.json())
            except: print("Server response (text):", detail.text)

Using device: cuda


In [4]:
best_score, epoch_log = train("train.npz")
save_report(best_score, epoch_log)
submit(API_KEY, MODEL_PATH)

Epoch   1/100 | Loss: 2.2912 | Train Adv Acc: 0.276 | Val Clean: 0.297 | Val Robust: 0.285 | Score: 0.291
  ✓ Saved best model (score=0.2910) → /kaggle/working/model.pt


Epoch   2/100 | Loss: 1.8227 | Train Adv Acc: 0.291


Epoch   3/100 | Loss: 1.7968 | Train Adv Acc: 0.290


Epoch   4/100 | Loss: 1.7775 | Train Adv Acc: 0.298


Epoch   5/100 | Loss: 1.7769 | Train Adv Acc: 0.299 | Val Clean: 0.308 | Val Robust: 0.233 | Score: 0.271


...

Epoch  96/100 | Loss: 1.3390 | Train Adv Acc: 0.471


Epoch  97/100 | Loss: 1.3368 | Train Adv Acc: 0.473


Epoch  98/100 | Loss: 1.3373 | Train Adv Acc: 0.473


Epoch  99/100 | Loss: 1.3352 | Train Adv Acc: 0.473


Epoch 100/100 | Loss: 1.3340 | Train Adv Acc: 0.473 | Val Clean: 0.684 | Val Robust: 0.444 | Score: 0.564

Done. Best score: 0.5638
Report saved → /kaggle/working/report.txt
Successfully submitted.
Server response: {'submission_id': 2666, 'status': 'success', 'message': 'Submission evaluated successfully! Check the leaderboard to see your score.'}
